## Código Python para o Modelo de Otimização Estocástica

In [8]:
import pulp
import random

# Fixa a semente do gerador aleatorio para resultados reprodutiveis
random.seed(42)

# 1. DEFINICAO DOS DADOS BASE (Extraidos do Artigo - Situacao I)
num_strata = 5
base_revenue_c = [404.40, 286.5, 183.3, 102.57, 127.0] # Receita liquida
area_limits_A = [21.66, 14.17, 11.33, 6.34, 7.83] # Limites de area
base_productivity_r = [50.57, 53.08, 43.09, 40.71, 38.69] # Produtividade de lenha
firewood_demand_min = 2300 # Demanda minima de lenha
base_labor_m = [87.5, 68.3, 58.0, 56.5, 48.6] # Mao de obra necessaria
base_labor_M_max = 3500 # Disponibilidade de mao de obra

# 2. GERACAO DE CENARIOS ESTOCASTICOS
num_scenarios = 50
probabilities = [1.0 / num_scenarios] * num_scenarios

# Custos de Recurso (suposicoes)
q_mo = 10.0      # Custo por hora de mao de obra extra
q_deficit = 50.0 # Penalidade por unidade de lenha nao entregue

scenarios = []
for i in range(num_scenarios):
    # Gera parametros incertos para este cenario
    scenario_c = [c * random.uniform(0.85, 1.15) for c in base_revenue_c]
    scenario_r = [r * random.uniform(0.85, 1.15) for r in base_productivity_r]
    scenario_M_max = base_labor_M_max * random.uniform(0.90, 1.10)

    scenarios.append({
        "c": scenario_c,
        "r_lenha": scenario_r,
        "M_max": scenario_M_max
    })

# 3. MODELAGEM COM PULP
# Inicializa o modelo
model = pulp.LpProblem("GestaoFlorestalEstocastica", pulp.LpMaximize)

# Variaveis de Primeiro Estagio
E = [pulp.LpVariable(f'E_{j+1}', lowBound=0) for j in range(num_strata)]

# Variaveis de Segundo Estagio (Recurso)
y_mo = pulp.LpVariable.dicts("y_mo", range(num_scenarios), lowBound=0)
y_deficit = pulp.LpVariable.dicts("y_deficit", range(num_scenarios), lowBound=0)

# Funcao Objetivo: Maximizar lucro esperado
expected_revenue = pulp.lpSum(
    probabilities[s] * pulp.lpSum(scenarios[s]['c'][j] * E[j] for j in range(num_strata))
    for s in range(num_scenarios)
)
expected_recourse_cost = pulp.lpSum(
    probabilities[s] * (q_mo * y_mo[s] + q_deficit * y_deficit[s])
    for s in range(num_scenarios)
)
model += expected_revenue - expected_recourse_cost, "Lucro_Esperado_Liquido"

# Restricoes de Primeiro Estagio
for j in range(num_strata):
    model += E[j] <= area_limits_A[j], f'Limite_Area_E{j+1}'

# Restricoes de Segundo Estagio (para cada cenario)
for s in range(num_scenarios):
    # Restricao de Mao de Obra
    labor_needed = pulp.lpSum(base_labor_m[j] * E[j] for j in range(num_strata))
    model += labor_needed <= scenarios[s]['M_max'] + y_mo[s], f'Restricao_MO_Cenario_{s+1}'

    # Restricao de Demanda de Lenha
    firewood_produced = pulp.lpSum(scenarios[s]['r_lenha'][j] * E[j] for j in range(num_strata))
    model += firewood_produced >= firewood_demand_min - y_deficit[s], f'Restricao_Lenha_Cenario_{s+1}'

# 4. RESOLUCAO E APRESENTACAO DOS RESULTADOS
model.solve()

# Calcula os custos esperados extraindo os valores otimos
expected_mo_cost_val = sum(probabilities[s] * q_mo * y_mo[s].varValue for s in range(num_scenarios))
expected_deficit_cost_val = sum(probabilities[s] * q_deficit * y_deficit[s].varValue for s in range(num_scenarios))
expected_gross_revenue_val = pulp.value(model.objective) + expected_mo_cost_val + expected_deficit_cost_val

# Exibe os resultados
print(f"Status da Solução: {pulp.LpStatus[model.status]}")
print("-" * 35)

print("ANÁLISE FINANCEIRA ESPERADA:")
print(f"Receita Bruta Esperada: US$ {expected_gross_revenue_val:.2f}")
print(f"Custo Esperado de Mão de Obra Extra: US$ {expected_mo_cost_val:.2f}")
print(f"Penalidade Esperada por Déficit: US$ {expected_deficit_cost_val:.2f}")
print(f"Lucro Líquido Esperado: US$ {pulp.value(model.objective):.2f}")
print("-" * 35)

print("Valores Ótimos das Variáveis de 1º Estágio (Colheita):")
for j in range(num_strata):
    print(f"E_{j+1} = {E[j].varValue:.4f} ha")
print("-" * 35)

# Ocultando a impressão de todas as variaveis y_mo e y_deficit e slacks para manter o log limpo,
# mas se quiser ver tudo, basta manter o loop original de model.variables() e model.constraints()

Status da Solução: Optimal
-----------------------------------
ANÁLISE FINANCEIRA ESPERADA:
Receita Bruta Esperada: US$ 15072.33
Custo Esperado de Mão de Obra Extra: US$ 1622.44
Penalidade Esperada por Déficit: US$ 180.38
Lucro Líquido Esperado: US$ 13269.51
-----------------------------------
Valores Ótimos das Variáveis de 1º Estágio (Colheita):
E_1 = 21.6600 ha
E_2 = 14.1700 ha
E_3 = 11.3300 ha
E_4 = 0.0000 ha
E_5 = 2.1721 ha
-----------------------------------
